<div>
<img src="./images/sunpy_logo.png" width="500" align="left"/>
</div>

## Welcome, what we're doing for the next 2.5 hours?

Four notebooks, all built around one real solar event (an M-class flare on **2022-04-02**):

| # | Notebook | What it covers |
|---|---|---|
| 01 | this one | `Fido`: how to query and download data from any solar archive |
| 02 | `2_data_containers.ipynb` | `Map` and `TimeSeries`: the containers sunpy uses |
| 03 | `3_coordinates_framework.ipynb` | `SkyCoord` and solar frames: the glue |
| 04 | `4_example_workflow.ipynb` | End-to-end: GOES → AIA → LASCO → Solar Orbiter |

### What is SunPy?

SunPy is the **open-source Python ecosystem for solar physics**. It has three layers:

- **`sunpy` core**: `Fido` (the unified data search/fetch interface), `Map` and `TimeSeries` (the data containers), and a coordinate framework that knows about the Sun and the spacecraft observing it.
- **Affiliated packages**: focused tools that plug into the core. Today we'll meet `aiapy`, `sunpy-soar`, `sunkit-instruments`, and `sunkit-image`. There are ~25 affiliated packages in total — see [sunpy.org/project/affiliated.html](https://sunpy.org/project/affiliated.html).
- **A community** : open-source, NumFOCUS sponsored, several active maintainers (and could be you too!), used in 1000+ refereed papers. PRs and issues welcome! We'd love to have more people involved!

### Why programmatic access?

JHelioviewer (Wednesday's session) is brilliant for visual exploration, perfect for browsing the corona, finding interesting events, taking screenshots for talks. But the moment you want to *do science* with the data, for example measure a flux, fit a CME front, build a statistical sample of flares, co-align a magnetogram with a coronal image, reproject SDO's view into Solar Orbiter's frame, query the GOES catalogue for every M-class event in 2024 and analyse them all the same way, none of that is a button you can click. You write code. SunPy gives youthe building blocks.                                  A bonus: your analysis becomes a script. That means it's reproducible (someone else can run it), iterative (you can re-run it    
  when an instrument calibration is updated), and shareable (PR it into a paper, a thesis, or a colleague's project).

Today with these sunpy intro notebooks give you the tooling. The lectures later in the week (flares Tue, solar wind Wed, CMEs Wed) give you the why.

## Resources to keep open

- [sunpy documentation](https://docs.sunpy.org/en/stable/): the place to look things up
- [sunpy gallery](https://docs.sunpy.org/en/stable/generated/gallery/index.html): short example scripts for common tasks
- [Matrix chat](https://openastronomy.element.io/#/room/#sunpy:openastronomy.org): ask questions, get help

> **Running on Google Colab?** Each notebook fetches its own data. On Colab, files downloaded in one notebook *don't* persist to other notebooks (each session has its own filesystem), so each notebook re-fetches what it needs, slower but works. On a local install, the `data/` directory is shared across notebooks, so files only download once.

## 0. Setup if using google colab

In [71]:
import sys
if "google.colab" in sys.modules:
    !wget -q https://raw.githubusercontent.com/hayesla/ESPD_2026/main/requirements.txt
    %pip install --quiet -r requirements.txt
    print("Colab install complete. If imports below fail, do Runtime > Restart runtime.")

# 1. Searching and downloading data with sunpy



In this notebook, an introduction to how you can search for and download data with sunpy. We will begin with an intoduction to `astropy.units` (which are used throughout the sunpy ecosystem), and then look about how to use `Fido` and build queries for data. In particular, this notebook will look at the following:

1. Introduction to `astropy.units`
2. Overview of `Fido` 
3. Constructing a data search query and inspecting it
4. More complex queries and the HEK
5. Extending Fido - the SOAR archive

In [2]:
import astropy.units as u

from sunpy.net import Fido, attrs as a
from sunpy.time import parse_time

# sunpy_soar is an affiliated package of SunPy 
# and registers the SOAR to be searched by Fido
import sunpy_soar

import numpy as np

> **Heads-up**: this tutorial needs `sunpy >= 7.1`. If you see an older version, update with `pip install --upgrade "sunpy[all]"` (or recreate the conda env from `environment.yml`).

## 1.1 Astropy Units - a quick overview
[`astropy.units`](https://docs.astropy.org/en/stable/units/) provides a means to deal with and handle numbers/arrays etc that have an associated physical quantity (e.g. km, seconds, Kelvin). Throughout SunPy, any physical input or outputs is an [`astropy.Quantity`](https://docs.astropy.org/en/stable/units/quantity.html#quantity). Lets look at how we can create and convert between astropy units. Above we have imported `astropy.units` as `u`

In [3]:
distance_in_km = 10*u.km

In [4]:
distance_in_km

<Quantity 10. km>

In [5]:
distance_in_km.unit

Unit("km")

In [6]:
distance_in_km.value

np.float64(10.0)

We can convert between equivalent units

In [7]:
distance_in_km.cgs

<Quantity 1000000. cm>

In [8]:
distance_in_km.to(u.parsec)

<Quantity 3.24077929e-13 pc>

In [9]:
distance_in_km.to(u.Mm)

<Quantity 0.01 Mm>

However you can only convert between physical units that make sense for example:

In [10]:
#distance_in_km.to(u.second)

In [11]:
time_in_sec = 60*u.s

In [12]:
(distance_in_km/time_in_sec).unit

Unit("km / s")

In [13]:
(10*u.Angstrom).to(u.nm)

<Quantity 1. nm>

> **Why this matters**: sunpy uses this framework thoughout, and `Fido` queries take Quantities, not bare numbers. `Wavelength(171 * u.angstrom)` works; `Wavelength(171)` does not.

# 1.2 Overview of sunpy's Fido Unified Downloader
### The problem

Solar data lives in many places:
- **SDO/AIA, HMI** at the *Joint Science Operations Center* (JSOC) at Stanford and are accessible through interfaces such as the *Virtual Solar Observatory* (VSO)
- **SOHO/LASCO**, MDI, EIT at the Naval Research Lab (also via VSO).
- **Solar Orbiter** data (EUI, MAG, STIX, SPICE, …) at the *Solar Orbiter Archive* (SOAR).
- **In-situ time series** (Wind, ACE, DSCOVR, …) at NASA's *CDAWeb*.
- **Flare/CME event lists** at the *Heliophysics Event Knowledgebase* (HEK).

Each archive has its own API, its own quirks. Writing analysis code that hits all of them directly is painful.

### The solution

**`Fido`** is a *unified* search/fetch interface. You write what you want; `Fido` figures out which archive serves that data and goes there.

For more information about Fido and how to use it check out the documentation on our website: https://docs.sunpy.org/en/stable/tutorial/acquiring_data/index.html

Fido offers access to data available through:

* VSO (Virtual Solar Observatory)
* JSOC (through drms)
* Individual data providers from web accessible sources (http, ftp, etc)
* CDAWeb
* HEK
* HELIO
  
As described here Fido provides access to many sources of data through different clients, these clients can be defined inside sunpy or in other packages (e.g. DKIST data can be accessed using Fido through [DKIST User Tools](https://docs.dkist.nso.edu/projects/python-tools/en/latest/tutorial/2_search_and_asdf_download.html)).

### Clients, what they are, why you don't talk to them directly

Under the hood, each archive has a **client**, a small adapter class that translates a `Fido` query into that archive's native API. There are clients for VSO, JSOC, SOAR, CDAWeb, HEK, and a few more. You don't talk to them directly, but it's worth knowing they exist:

- Two archives sometimes serve overlapping data (AIA is at both VSO and JSOC), search results show rows from each.
- Some attrs are *client-specific*: `a.jsoc.Notify`, `a.goes.SatelliteNumber`, `a.soar.Product`, `a.cdaweb.Dataset`. These live under the namespace of the client they belong to.

#### Importantly, Solar Orbiter data can be accessed through the client defined in the `sunpy_soar` affiliated package.
The SOAR client is registered once we install `sunpy_soar` above. Without installing it, it wont be registered within Fido.

Lets first inspect the clients that are available through Fido:

In [14]:
Fido

Client,Description
CDAWEBClient,Provides access to query and download from the Coordinated Data Analysis Web (CDAWeb).
ADAPTClient,Provides access to the ADvanced Adaptive Prediction Technique (ADAPT) products of the National Solar Observatory (NSO).
AIASynopsisClient,"A client for retrieving AIA ""synoptic"" data from the JSOC."
EVEClient,Provides access to Level 0CS Extreme ultraviolet Variability Experiment (EVE) data.
GBMClient,Provides access to data from the Gamma-Ray Burst Monitor (GBM) instrument on board the Fermi satellite.
XRSClient,Provides access to several GOES XRS files archive.
SUVIClient,Provides access to data from the GOES Solar Ultraviolet Imager (SUVI).
GONGClient,Provides access to the Magnetogram products of NSO-GONG synoptic Maps.
LYRAClient,Provides access to the LYRA/Proba2 data archive.
NOAAIndicesClient,Provides access to the NOAA solar cycle indices.


### Using attributes to search for data with Fido

Sunpy uses specified **attributes** to search for data using Fido. The range of these attributes is located in the `attrs` submodule. These `attr` parameters can be combined together to construct data search queries, such as searching over a certain time period, for data from a certain instrument with a certain wavelength etc.

Different clients and provides will have client-specific attributes, but the core attributes are:

* `a.Time`
* `a.Instrument`
* `a.Wavelength`


Lets look at how these attributes work in more detail.

First we can look at `a.Time`, which is used to specify the timerange of a query.


### Anatomy of a Fido query

A query is built from `attrs`, small descriptors of "what you want":

```python
from sunpy.net import attrs as a

a.Time('2022-04-02 12:00', '2022-04-02 14:00')   # time range
a.Instrument('AIA')                              # instrument
a.Wavelength(171 * u.angstrom)                   # wavelength
a.Sample(1 * u.min)                              # cadence (downsample)
```

Combine with `&` (AND) and `|` (OR), and pass the whole thing to `Fido.search`.

> **Two ways to write `a.Instrument`**: `a.Instrument('AIA')` (string form) and `a.Instrument.aia` (attribute form) are equivalent. The attribute form is autocompleted by your editor and validated against the list of known instruments, so we'll prefer it.

In [15]:
a.Time("2022-04-02 12:00", "2022-04-02 15:00")

<sunpy.net.attrs.Time(2022-04-02 12:00:00.000, 2022-04-02 15:00:00.000)>

We can inspect the instrument attribute to see what instrument `attrs` are currently supported through sunpy. Here we can see the instrument name (i.e. the name to be passed to the `a.Instrument` attribute, the client from which the data is available to access, and the full name of the instrument.)

In [16]:
a.Instrument

Attribute Name,Client,Full Name,Description
adapt,ADAPT,ADAPT,ADvanced Adaptive Prediction Technique.
aia,AIASynopsis,AIA,Data from the Atmospheric Imaging Assembly instrument.
aia,VSO,AIA,Atmospheric Imaging Assembly
bcs,VSO,BCS,Bragg Crystal Spectrometer
be_continuum,VSO,BE-Continuum,INAF-OACT Barra Equatoriale Continuum Instrument
be_halpha,VSO,BE-Halpha,INAF-OACT Barra Equatoriale Hα Instrument
bigbear,VSO,Big Bear,"Big Bear Solar Observatory, California TON and GONG+ sites"
c1,VSO,C1,Coronagraph #1 (H alpha line)
c2,VSO,C2,Coronagraph #2 (HeI and FeXIII lines)
caii,VSO,CAII,Kanzelhöhe Ca II k Instrument


sunpy also now provides tab completion to auto-fill the attribute name

In [17]:
a.Instrument.eit

<sunpy.net.attrs.Instrument(EIT: Extreme ultraviolet Imaging Telescope) object at 0x12fcd1400>

To search for certain wavelengths, we need to specify the input as an `astropy.Quantity` which is a the combination of a value and an associated unit. This is something is universal in the sunpy stack - that every physical input/output is a `Quantity`.

In [18]:
a.Wavelength(17.1*u.angstrom)

<sunpy.net.attrs.Wavelength(17.1, 17.1, 'Angstrom')>

In [19]:
import sunpy

## 3. Constructing a search query
 ### A simple query

Lets create a simple query to search for data from AIA over a particular time period

In [20]:
result = Fido.search(a.Time("2022-04-02 12:00", "2022-04-02 15:00"), 
                     a.Instrument("AIA"))

### Reading a Fido search result

The thing returned is a `UnifiedResponse`. The display groups results by *client*, you'll see headings like `the cdaweb client` or `the VSO client`. Each row is one file you *could* download.

**Two-index addressing**: `result[i, j]` selects client `i`, row `j` within that client.
- `result[0, 0]` = first file from the first client
- `result[0, :3]` = first 3 files from the first client
- `result[0]` = entire first client's table

In [21]:
result

Start Time,End Time,Source,Instrument,Wavelength,Provider,Physobs,Wavetype,Extent Width,Extent Length,Extent Type,Size,Extra Flags
,,,,Angstrom,,,,,,,Mibyte,
Time,Time,str3,str3,float64[2],str4,str9,str6,str4,str4,str8,float64,str1
2022-04-02 12:00:00.000,2022-04-02 12:09:13.000,SDO,AIA,335.0 .. 335.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,3038.47656,S
2022-04-02 12:00:04.000,2022-04-02 12:09:17.000,SDO,AIA,193.0 .. 193.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,3038.47656,S
2022-04-02 12:00:05.000,2022-04-02 12:00:06.000,SDO,AIA,4500.0 .. 4500.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844,S
2022-04-02 12:00:05.000,2022-04-02 12:09:18.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,3038.47656,S
2022-04-02 12:00:06.000,2022-04-02 12:09:19.000,SDO,AIA,131.0 .. 131.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,3038.47656,S
2022-04-02 12:00:09.000,2022-04-02 12:09:10.000,SDO,AIA,211.0 .. 211.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,2973.82812,S
2022-04-02 12:00:09.000,2022-04-02 12:09:22.000,SDO,AIA,171.0 .. 171.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,3038.47656,S
2022-04-02 12:00:11.000,2022-04-02 12:09:12.000,SDO,AIA,94.0 .. 94.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,2973.82812,S


Now lets make our query a bit more specific, say, say we only want one wavelength band from AIA. This can be achieved by specifying the `Wavelength` attribute within the search. The `a.Wavelength` attribute is passed as an `astropy.Quantity`:

In [22]:
result = Fido.search(a.Time("2022-04-02 12:00", "2022-04-02 15:00"), 
                     a.Instrument("AIA"), 
                     a.Wavelength(304*u.angstrom))

In [23]:
result

Start Time,End Time,Source,Instrument,Wavelength,Provider,Physobs,Wavetype,Extent Width,Extent Length,Extent Type,Size
,,,,Angstrom,,,,,,,Mibyte
Time,Time,str3,str3,float64[2],str4,str9,str6,str4,str4,str8,float64
2022-04-02 12:00:05.000,2022-04-02 12:00:06.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844
2022-04-02 12:00:17.000,2022-04-02 12:00:18.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844
2022-04-02 12:00:29.000,2022-04-02 12:00:30.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844
2022-04-02 12:00:41.000,2022-04-02 12:00:42.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844
2022-04-02 12:00:53.000,2022-04-02 12:00:54.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844
2022-04-02 12:01:05.000,2022-04-02 12:01:06.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844
2022-04-02 12:01:17.000,2022-04-02 12:01:18.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844
2022-04-02 12:01:29.000,2022-04-02 12:01:30.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844


We can further specify this query by choosing the cadence (time-sampling) of the data we want to search for and download. This can be achieved by using the Sample attribute. Similar to the Wavelength attributes, this needs to be an astropy Quantity. Lets further specify the search above to only search for data with a cadence of 10 minutes.

In [24]:
result = Fido.search(a.Time("2022-04-02 12:00", "2022-04-02 15:00"), 
                     a.Instrument("AIA"), 
                     a.Wavelength(171*u.angstrom),
                     a.Sample(10*u.min))

In [25]:
len(result[0])

18

## 1.3 Downloading the data

Now we can show how data that is queried above can be downloaded. Once the data you have searched for (and filtered etc) is constructed into a query using `Fido.search`, you can then easily download them using `Fido.fetch`.

The data is downloaded via asynchronous and parallel download streams (via parfive), and also allows for failed data downloads to be recognized so that files can be re-requested if not downloaded.

Lets now look at how a `UnifiedResponse` from a `Fido.search` can be passed to `Fido.fetch` to download the data

### Fetching, actually downloading the data from the search result

`Fido.fetch(...)` does the download. You can use `path=` to tell sunpy where to download the files to. If you dont specify it will download them to the location set in the sunpy config file (typically `sunpy/data`).
The `path=` template uses placeholders:
- `{file}`: replaced with the filename suggested by the client
- other available placeholders: `{instrument}`, `{source}`, `{detector}`, …

We'll save into the project-local `../data/` directory so files cache between notebooks (on local installs).

In [26]:
files = Fido.fetch(result, site="NSO")

Files Downloaded:   0%|          | 0/18 [00:00<?, ?file/s]

These files are downloaded to a local location set in the sunpy.config.file, which by default is ~/sunpy/data/. Fido.fetch returns a parfile.Results object which gives the path to where the files are downloaded to

In [28]:
print(files[0])

/Users/laurahayes/sunpy/data/aia.lev1.171A_2022_04_02T12_00_09.35Z.image_lev1.fits


You can also define what directory you want the files to be saved to by passing the directory path to the path keyword in Fido.fetch. For example, I want to download these files to a local directory `./AIA/<name_of_file>`

In [27]:
Fido.fetch(result, path="./data/{file}", site="NSO")

Files Downloaded:   0%|          | 0/18 [00:00<?, ?file/s]

Exception ignored in: <function BaseEventLoop.__del__ at 0x10560e7a0>
Traceback (most recent call last):
  File "/Users/laurahayes/miniforge3/envs/espd-sunpy-2026/lib/python3.12/asyncio/base_events.py", line 732, in __del__
    self.close()
  File "/Users/laurahayes/miniforge3/envs/espd-sunpy-2026/lib/python3.12/asyncio/unix_events.py", line 71, in close
    self.remove_signal_handler(sig)
  File "/Users/laurahayes/miniforge3/envs/espd-sunpy-2026/lib/python3.12/asyncio/unix_events.py", line 160, in remove_signal_handler
    signal.signal(sig, handler)
  File "/Users/laurahayes/miniforge3/envs/espd-sunpy-2026/lib/python3.12/signal.py", line 58, in signal
    handler = _signal.signal(_enum_to_int(signalnum), _enum_to_int(handler))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: signal only works in main thread of the main interpreter


aia.lev1.171A_2022_04_02T12_30_09.35Z.image_lev1.fits:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

aia.lev1.171A_2022_04_02T12_00_09.35Z.image_lev1.fits:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

aia.lev1.171A_2022_04_02T12_40_09.35Z.image_lev1.fits:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

...

aia.lev1.171A_2022_04_02T14_20_09.35Z.image_lev1.fits:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

aia.lev1.171A_2022_04_02T14_30_09.35Z.image_lev1.fits:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

aia.lev1.171A_2022_04_02T14_40_09.35Z.image_lev1.fits:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

aia.lev1.171A_2022_04_02T14_50_09.35Z.image_lev1.fits:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

['data/aia.lev1.171A_2022_04_02T12_00_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T12_10_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T12_20_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T12_30_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T12_40_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T12_50_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T13_00_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T13_10_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T13_20_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T13_30_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T13_40_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T13_50_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T14_00_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T14_10_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T14_20_09.35Z.image_lev1.fits', 'data/aia.lev1.171A_2022_04_02T14_30_09.35Z.image_lev1.fits', 'data/a

## 1. 4 More complex queries

In addition to making a query to one client for one instrument, `Fido` allows the flexibility to search for data from multiple instruments, wavelengths, times etc, even when the data is being obtained through different clients.

This query can be constructed by using the pipe `|` operator, which joins queries together just like the OR operator.

Lets now make a query that searches for both GOES/XRS and AIA data over a particular time period

In [28]:
result = Fido.search(a.Time("2022-04-02 12:00", "2022-04-02 15:00"), 
                     a.Instrument.xrs  | (a.Instrument.aia & a.Wavelength(304*u.angstrom) & a.Sample(10*u.min)))

In [29]:
len(result)

2

In [30]:
result

<sunpy.net.fido_factory.UnifiedResponse object at 0x12fc8bb30>
Results from 2 Providers:

4 Results from the XRSClient:
Source: <8: https://umbra.nascom.nasa.gov/goes/fits 
8-15: https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/ 
16-17: https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/

       Start Time               End Time        Instrument  Physobs   Source Provider Resolution SatelliteNumber
----------------------- ----------------------- ---------- ---------- ------ -------- ---------- ---------------
2022-04-02 00:00:00.000 2022-04-02 23:59:59.999        XRS irradiance   GOES     NOAA      flx1s              16
2022-04-02 00:00:00.000 2022-04-02 23:59:59.999        XRS irradiance   GOES     NOAA      avg1m              16
2022-04-02 00:00:00.000 2022-04-02 23:59:59.999        XRS irradiance   GOES     NOAA      flx1s              17
2022-04-02 00:00:00.000 2022-04-02 23:59:59.999        XRS irradiance   GOES     NOAA      avg1m              17

18 Results from the VSOClient:
Source: https://sdac.virtualsolar.org/cgi/search
Data retrieval status: http://docs.virtualsolar.org/wiki/VSOHealthReport
Total estimated size: 1.22 Gbyte

       Start Time               End Time        Source Instrument   Wavelength   Provider  Physobs  Wavetype Extent Width Extent Length Extent Type   Size  
                                                                     Angstrom                                                                        Mibyte 
----------------------- ----------------------- ------ ---------- -------------- -------- --------- -------- ------------ ------------- ----------- --------
2022-04-02 12:00:05.000 2022-04-02 12:00:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 12:10:05.000 2022-04-02 12:10:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 12:20:05.000 2022-04-02 12:20:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 12:30:05.000 2022-04-02 12:30:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 12:40:05.000 2022-04-02 12:40:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 12:50:05.000 2022-04-02 12:50:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 13:00:05.000 2022-04-02 13:00:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 13:10:05.000 2022-04-02 13:10:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 13:20:05.000 2022-04-02 13:20:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 13:30:05.000 2022-04-02 13:30:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 13:40:05.000 2022-04-02 13:40:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 13:50:05.000 2022-04-02 13:50:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 14:00:05.000 2022-04-02 14:00:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 14:10:05.000 2022-04-02 14:10:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096          4096    FULLDISK 64.64844
2022-04-02 14:20:05.000 2022-04-02 14:20:06.000    SDO        AIA 304.0 .. 304.0     JSOC intensity   NARROW         4096  

In [31]:
result[0]

Start Time,End Time,Instrument,Physobs,Source,Provider,Resolution,SatelliteNumber
Time,Time,str3,str10,str4,str4,str5,int64
2022-04-02 00:00:00.000,2022-04-02 23:59:59.999,XRS,irradiance,GOES,NOAA,flx1s,16
2022-04-02 00:00:00.000,2022-04-02 23:59:59.999,XRS,irradiance,GOES,NOAA,avg1m,16
2022-04-02 00:00:00.000,2022-04-02 23:59:59.999,XRS,irradiance,GOES,NOAA,flx1s,17
2022-04-02 00:00:00.000,2022-04-02 23:59:59.999,XRS,irradiance,GOES,NOAA,avg1m,17


In [32]:
result.all_colnames

['End Time',
 'Instrument',
 'Physobs',
 'Provider',
 'Resolution',
 'SatelliteNumber',
 'Source',
 'Start Time',
 'url']

Lets download the GOES XRS data first

In [33]:
Fido.fetch(result[0, 0], path="./data/{file}")

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

['data/sci_xrsf-l2-flx1s_g16_d20220402_v2-2-1.nc']

Now lets say we only want to download one AIA file at a particular time, we can also search the table for certain conditions. Lets say we just want the file that closest to 2022-04-02 13:00. 

In [34]:
(np.abs(result[1]["Start Time"] - parse_time("2022-04-02 13:00"))).argmin()

np.int64(6)

In [55]:
result[1, 6]

Start Time,End Time,Source,Instrument,Wavelength,Provider,Physobs,Wavetype,Extent Width,Extent Length,Extent Type,Size,fileid
,,,,Angstrom,,,,,,,Mibyte,
Time,Time,str3,str3,float64[2],str4,str9,str6,str4,str4,str8,float64,str24
2022-04-02 13:00:05.000,2022-04-02 13:00:06.000,SDO,AIA,304.0 .. 304.0,JSOC,intensity,NARROW,4096,4096,FULLDISK,64.64844,aia__lev1:304:1427979644


In [56]:
Fido.fetch(result[1, 6], path="./data/", site="NSO")

Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

aia.lev1.304A_2022_04_02T13_00_05.13Z.image_lev1.fits:   0%|          | 0.00/7.50M [00:00<?, ?B/s]

['data/aia.lev1.304A_2022_04_02T13_00_05.13Z.image_lev1.fits']

---
## 1.5. Other and external clients

Some archives have their own dedicated `Fido` plugin packages. The most important for solar physics today:

- **CDAWeb** (built into sunpy): for in-situ time series
- **HEK** (built into sunpy): for event lists
- **`sunpy_soar`**: the Solar Orbiter Archive client

### Solar Orbiter Archive (SOAR)

`sunpy_soar` is an affiliated SunPy package that registers the Solar Orbiter Archive (SOAR) client with `Fido` automatically, just `import sunpy_soar` once and `a.soar.*` attrs become available.

## SOAR archive searching using sunpy!

In [57]:
import sunpy_soar

Note that after importing `sunpy_soar`, the SOAR is now listed as a client that `Fido` will search.

In [58]:
Fido

Client,Description
CDAWEBClient,Provides access to query and download from the Coordinated Data Analysis Web (CDAWeb).
ADAPTClient,Provides access to the ADvanced Adaptive Prediction Technique (ADAPT) products of the National Solar Observatory (NSO).
AIASynopsisClient,"A client for retrieving AIA ""synoptic"" data from the JSOC."
EVEClient,Provides access to Level 0CS Extreme ultraviolet Variability Experiment (EVE) data.
GBMClient,Provides access to data from the Gamma-Ray Burst Monitor (GBM) instrument on board the Fermi satellite.
XRSClient,Provides access to several GOES XRS files archive.
SUVIClient,Provides access to data from the GOES Solar Ultraviolet Imager (SUVI).
GONGClient,Provides access to the Magnetogram products of NSO-GONG synoptic Maps.
LYRAClient,Provides access to the LYRA/Proba2 data archive.
NOAAIndicesClient,Provides access to the NOAA solar cycle indices.


In [59]:
eui_query = Fido.search(a.Time("2022-04-02 12:00", "2022-04-02 15:00"), 
                        a.soar.Product("EUI-FSI174-IMAGE"), 
                        a.Level(2))

In [60]:
eui_query

Instrument,Data product,Level,Start time,End time,Data item ID,Filename,Filesize,SOOP Name,Detector,Wavelength
,,,,,,,Mbyte,,,
str3,str16,str2,str23,str23,str43,str52,float64,str30,str3,float64
EUI,eui-fsi174-image,L2,2022-04-02 12:00:45.225,2022-04-02 12:00:55.225,solo_L2_eui-fsi174-image_20220402T120045225,solo_L2_eui-fsi174-image_20220402T120045225_V02.fits,5.38,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0
EUI,eui-fsi174-image,L2,2022-04-02 12:10:45.226,2022-04-02 12:10:55.226,solo_L2_eui-fsi174-image_20220402T121045226,solo_L2_eui-fsi174-image_20220402T121045226_V02.fits,5.285,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0
EUI,eui-fsi174-image,L2,2022-04-02 12:20:45.227,2022-04-02 12:20:55.227,solo_L2_eui-fsi174-image_20220402T122045227,solo_L2_eui-fsi174-image_20220402T122045227_V02.fits,5.377,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0
EUI,eui-fsi174-image,L2,2022-04-02 12:30:45.228,2022-04-02 12:30:55.228,solo_L2_eui-fsi174-image_20220402T123045228,solo_L2_eui-fsi174-image_20220402T123045228_V02.fits,5.371,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0
EUI,eui-fsi174-image,L2,2022-04-02 12:40:45.229,2022-04-02 12:40:55.229,solo_L2_eui-fsi174-image_20220402T124045229,solo_L2_eui-fsi174-image_20220402T124045229_V02.fits,5.314,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0
EUI,eui-fsi174-image,L2,2022-04-02 12:50:45.230,2022-04-02 12:50:55.230,solo_L2_eui-fsi174-image_20220402T125045230,solo_L2_eui-fsi174-image_20220402T125045230_V02.fits,5.334,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0
EUI,eui-fsi174-image,L2,2022-04-02 13:00:45.231,2022-04-02 13:00:55.231,solo_L2_eui-fsi174-image_20220402T130045231,solo_L2_eui-fsi174-image_20220402T130045231_V02.fits,5.397,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0
EUI,eui-fsi174-image,L2,2022-04-02 13:10:45.269,2022-04-02 13:10:55.269,solo_L2_eui-fsi174-image_20220402T131045269,solo_L2_eui-fsi174-image_20220402T131045269_V02.fits,5.311,R_SMALL_MRES_MCAD_AR-Long-Term,FSI,174.0


In [61]:
Fido.fetch(eui_query, path="./data/{file}")

Files Downloaded:   0%|          | 0/18 [00:00<?, ?file/s]

['data/solo_L2_eui-fsi174-image_20220402T120045225_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T121045226_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T122045227_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T123045228_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T124045229_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T125045230_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T130045231_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T131045269_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T132045233_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T133045235_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T134045261_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T135045237_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T140045238_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T141045275_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T142045240_V02.fits', 'data/solo_L2_eui-fsi174-image_20220402T143045241_V02.fits', 'data/solo_L2_eui-fsi17

We can also search for other data products, for example the Solar Orbiter MAG

In [62]:
mag_query = Fido.search(a.Time("2022-04-02", "2022-04-05"), 
                        a.soar.Product("MAG-RTN-NORMAL-1-MINUTE"), 
                        a.Level(2))

In [63]:
mag_query

Instrument,Data product,Level,Start time,End time,Data item ID,Filename,Filesize,SOOP Name
,,,,,,,Mbyte,
str3,str23,str2,str23,str23,str40,str48,float64,object
MAG,mag-rtn-normal-1-minute,L2,2022-04-02 00:00:00.000,2022-04-03 00:00:00.000,solo_L2_mag-rtn-normal-1-minute_20220402,solo_L2_mag-rtn-normal-1-minute_20220402_V02.cdf,0.032,None
MAG,mag-rtn-normal-1-minute,L2,2022-04-03 00:00:00.000,2022-04-04 00:00:00.000,solo_L2_mag-rtn-normal-1-minute_20220403,solo_L2_mag-rtn-normal-1-minute_20220403_V02.cdf,0.032,None
MAG,mag-rtn-normal-1-minute,L2,2022-04-04 00:00:00.000,2022-04-05 00:00:00.000,solo_L2_mag-rtn-normal-1-minute_20220404,solo_L2_mag-rtn-normal-1-minute_20220404_V02.cdf,0.033,None
MAG,mag-rtn-normal-1-minute,L2,2022-04-05 00:00:00.000,2022-04-06 00:00:00.000,solo_L2_mag-rtn-normal-1-minute_20220405,solo_L2_mag-rtn-normal-1-minute_20220405_V02.cdf,0.033,None


In [64]:
mag_files = Fido.fetch(mag_query, path="./{instrument}/{file}")

Files Downloaded:   0%|          | 0/4 [00:00<?, ?file/s]

In [65]:
mag_files

['MAG/solo_L2_mag-rtn-normal-1-minute_20220402_V02.cdf', 'MAG/solo_L2_mag-rtn-normal-1-minute_20220403_V02.cdf', 'MAG/solo_L2_mag-rtn-normal-1-minute_20220404_V02.cdf', 'MAG/solo_L2_mag-rtn-normal-1-minute_20220405_V02.cdf']

> **Important SOAR attrs**: `a.soar.Product` (specific data product like `eui-fsi174-image`, `mag-rtn-normal`, `stx-l1-spectrogram`) and `a.Level` (1, 2, 3 for raw / calibrated / scientifically-validated). The full product list is on the [SOAR website](https://soar.esac.esa.int/).

# Accessing data from the CDAWeb with sunpy - which is very helpful for in-situ data

There is also a CDAWeb client within sunpy. CDAWeb data can be accessed when the `cdaweb.Dataset` attribute is provided to the search.

The data available from the SOAR is also available from the CDAWeb. You may be used to working with this (especially if you mainly work with in-situ observations), so lets go through how the data can also be accessed this way. This is handy, as you can also access many other in-situ measurements from this too.

In [66]:
res_cdaw = Fido.search(a.Time("2022-04-02", "2022-04-05"), 
                       a.cdaweb.Dataset('SOLO_L2_MAG-RTN-NORMAL-1-MINUTE'))

In [67]:
res_cdaw

Dataset,Start time,End time,URL
str31,str23,str23,str144
SOLO_L2_MAG-RTN-NORMAL-1-MINUTE,2022-04-02 00:00:29.000,2022-04-02 23:59:30.000,https://cdaweb.gsfc.nasa.gov/sp_phys/data/solar-orbiter/mag/science/l2/rtn-normal-1-minute/2022/solo_l2_mag-rtn-normal-1-minute_20220402_v02.cdf
SOLO_L2_MAG-RTN-NORMAL-1-MINUTE,2022-04-03 00:00:29.000,2022-04-03 23:59:30.000,https://cdaweb.gsfc.nasa.gov/sp_phys/data/solar-orbiter/mag/science/l2/rtn-normal-1-minute/2022/solo_l2_mag-rtn-normal-1-minute_20220403_v02.cdf
SOLO_L2_MAG-RTN-NORMAL-1-MINUTE,2022-04-04 00:00:29.000,2022-04-04 23:59:30.000,https://cdaweb.gsfc.nasa.gov/sp_phys/data/solar-orbiter/mag/science/l2/rtn-normal-1-minute/2022/solo_l2_mag-rtn-normal-1-minute_20220404_v02.cdf


In [70]:
mag_cdaw_files = Fido.fetch(res_cdaw, path="./data/")

Files Downloaded:   0%|          | 0/3 [00:00<?, ?file/s]

## Accessing Metadata queries e.g. information from the HEK

As well as Fido providing an interface to search for data files that can be downloaded, Fido also allows you to query metadata. Currently Fido supports metadata searching from the HEK, HELIO and JSOC.

Similar to what we have seen so far, the search results of these clients are a UnifiedResponse object which can then be indexed and the QueryResponse table accessed like an astropy table. Lets look at an example of how we can use Fido to query the HEK.

Lets query for the active regions defined by SWPC over the past month. This can be done by using the HEK client specific attributes a.hek.attrs

In [69]:
from sunpy.net import Fido, attrs as a

In [49]:
hek_search = Fido.search(
    a.Time('2022-04-02 00:00', '2022-04-02 23:59'),
    a.hek.FL,                              # flares
    a.hek.OBS.Observatory == 'GOES',       # detected by GOES
    a.hek.FL.GOESCls > 'M1.0',             # M-class or stronger
)
hek_search['hek']

gs_thumburl,comment_count,hpc_bbox,frm_humanflag,hgc_coord,obs_levelnum,hpc_coord,event_npixels,gs_imageurl,ar_polarity,frm_paramset,hrc_coord,event_starttime,ar_mtwilsoncls,event_type,intensmin,fl_fluence,obs_meanwavel,frm_url,skel_chaincode,bound_chaincode,noposition,active,intensmax,frm_versionnumber,fl_halphaclass,area_uncert,obs_dataprepurl,hpc_geom,hgc_bbox,intensmedian,chaincodetype,obs_channelid,event_clippedspatial,ar_noaaclass,SOL_standard,event_avg_rating,eventtype,hpc_boundcc,event_mapurl,frm_contact,ar_penumbracls,intensmean,bound_ccstartc1,frm_name,area_atdiskcenter,frm_identifier,obs_observatory,event_description,boundbox_c2ur,obs_firstprocessingdate,boundbox_c2ll,frm_institute,hrc_bbox,refs_orig,ar_mcintoshcls,event_maskurl,bound_ccstartc2,gs_movieurl,event_score,skel_startc2,skel_startc1,fl_efoldtime,event_expires,hrc_boundcc,event_probability,hpc_heliosphere_corrected,intensvar,frm_daterun,hpc_y,hpc_x,search_instrument,ar_numspots,kb_archivdate,kb_archivist,intenstotal,sum_overlap_scores,hgs_boundcc,intensskew,obs_includesnrt,rasterscan,kb_archivid,search_frm_name,boundbox_c1ur,ar_noaanum,area_atdiskcenteruncert,boundbox_c1ll,event_importance_num_ratings,ar_compactnesscls,skel_curvature,event_testflag,event_c2error,hrc_r,skel_nsteps,hgs_y,obs_title,hgs_x,hcr_checked,frm_specificid,event_title,obs_instrument,event_c1error,revision,hpc_radius,event_endtime,event_importance,search_observatory,area_raw,concept,solar_object_locator,hgc_boundcc,fl_peakflux,hgc_x,hrc_a,event_peaktime,hgc_y,gs_galleryid,fl_goescls,hgs_coord,ar_zurichcls,bound_ccnsteps,intenskurt,event_clippedtemporal,hee_x,hee_y,hee_z,fl_peakem,rasterscantype,search_channelid,fl_peaktemp,hgs_bbox,obs_lastprocessingdate,refs,event_coord
,,"arcsec,arcsec",,"deg,deg",,"arcsec,arcsec",,,,,"solRad,deg,solRad",,,,,,cm,,"deg,deg","deg,deg",,,,,,,,,"deg,deg",,,,,,,,,"arcsec,arcsec",,,,,,,,,,,deg,,deg,,"solRad,deg,solRad",,,,,,,,,,,"solRad,deg,solRad",,,,,,,,,,,,,"deg,deg",,,,,,deg,,,deg,,,,,deg,,,,,,,,,,deg,,,,,,,,,"deg,deg",,,,,,,,"deg,deg",,,,,,,,,,,,"deg,deg",,,
str1,str1,SkyCoord,str5,SkyCoord,float64,SkyCoord,float64,str1,int64,str27,SkyCoord,Time,str2,str2,float64,float64,float64,str25,SkyCoord,SkyCoord,str4,str4,float64,float64,str2,float64,str2,str1,SkyCoord,float64,str2,str3,str2,str2,str30,object,str1,SkyCoord,str2,str23,str2,float64,float64,str4,float64,str4,str4,str2,float64,str2,float64,str61,SkyCoord,str1,str2,str2,float64,str1,str18,float64,float64,float64,str2,SkyCoord,float64,str5,float64,str19,float64,int64,str4,int64,str19,str19,float64,str1,SkyCoord,float64,str2,str2,str66,str4,float64,int64,float64,float64,str1,str2,float64,str5,float64,float64,int64,int64,str2,int64,str4,str2,str1,str4,float64,str1,str10,Time,float64,str4,float64,str5,str30,SkyCoord,float64,float64,int64,Time,int64,str1,str4,SkyCoord,str2,int64,float64,str2,object,object,str1,float64,str2,str3,float64,SkyCoord,str2,object[1],object
,0,"-1.675353999948129,-953.84328 .. -1.675353999948129,-953.84328",false,"22.408673,0.0",--,"0.0,109.084614",———,,--,"SSWIDL get_gev, ..., ngdc=0","0.113688287031148,90.0,1.0",2022-04-02 02:39:00.000,,FL,———,———,5e-08,http://www.swpc.noaa.gov/,"———,———","———,———",true,true,———,--,,———,,,"292.508673,-89.9 .. 292.508673,-89.9",———,,XRA,,,SOL2022-04-02T02:39:00L022C090,None,9,"———,———",,SWPC.Webmaster@noaa.gov,,———,———,SWPC,———,SWPC,GOES,,89.9000015258789,,-89.9000015258789,"U.S. Dept. of Commerce, NOAA, Space Weather Prediction Center","0.9941,269.899364,1.0 .. 0.9941,269.899364,1.0",,,,———,,0.6579999999999999,———,———,———,,"———,———,1.0",--,false,———,2022-04-02T00:00:00,109.084614,0,GOES,--,2022-05-25T16:27:54,autosubmission_swpc,———,0,"———,———",———,,,ivo://helio-informatics.org/FL_SWPC_20220525_162751_20220402023900,SWPC,89.9000015258789,0,———,-89.9000015258789,,,--,false,90.0,0.113688287031148,--,0,,0,true,,,GOES,90.0,1,109.084614,2022-04-02 03:07:00.000,--,GOES,———,Flare,SOL2022-04-02T02:39:00L022C090,"———,———",———,22.408673,0,2022-04-02 02:

In [50]:
hek_search["hek"]["event_starttime", "event_peaktime",
                               "event_endtime", "fl_goescls", "ar_noaanum", "frm_name"]

event_starttime,event_peaktime,event_endtime,fl_goescls,ar_noaanum,frm_name
Time,Time,Time,str4,int64,str4
2022-04-02 02:39:00.000,2022-04-02 02:56:00.000,2022-04-02 03:07:00.000,M2.9,0,SWPC
2022-04-02 12:56:00.000,2022-04-02 13:55:00.000,2022-04-02 14:44:00.000,M3.9,0,SWPC
2022-04-02 17:34:00.000,2022-04-02 17:44:00.000,2022-04-02 17:51:00.000,M4.3,0,SWPC


In [51]:
hek_search = Fido.search(a.Time("2022-04-02", "2022-04-03"), 
                         a.hek.CE)

In [52]:
hek_search[0][0]

gs_thumburl,comment_count,hpc_bbox,frm_humanflag,hgc_coord,obs_levelnum,hpc_coord,event_npixels,gs_imageurl,ar_polarity,frm_paramset,hrc_coord,event_starttime,ar_mtwilsoncls,event_type,intensmin,obs_meanwavel,frm_url,bound_chaincode,noposition,active,intensmax,cme_accel,frm_versionnumber,area_uncert,obs_dataprepurl,hpc_geom,hgc_bbox,intensmedian,chaincodetype,obs_channelid,event_clippedspatial,ar_noaaclass,event_avg_rating,eventtype,hpc_boundcc,event_mapurl,frm_contact,ar_penumbracls,intensmean,bound_ccstartc1,frm_name,area_atdiskcenter,frm_identifier,obs_observatory,event_description,boundbox_c2ur,obs_firstprocessingdate,boundbox_c2ll,frm_institute,hrc_bbox,refs_orig,ar_mcintoshcls,event_maskurl,bound_ccstartc2,gs_movieurl,event_score,cme_acceluncert,event_expires,hrc_boundcc,event_probability,hpc_heliosphere_corrected,intensvar,frm_daterun,cme_angularwidth,hpc_y,hpc_x,search_instrument,ar_numspots,kb_archivdate,kb_archivist,intenstotal,sum_overlap_scores,hgs_boundcc,intensskew,obs_includesnrt,rasterscan,cme_radiallinvelstddev,kb_archivid,search_frm_name,boundbox_c1ur,ar_noaanum,area_atdiskcenteruncert,boundbox_c1ll,event_importance_num_ratings,ar_compactnesscls,cme_massuncert,event_testflag,event_c2error,cme_mass,hrc_r,obs_title,hcr_checked,frm_specificid,event_title,obs_instrument,event_c1error,revision,hpc_radius,event_endtime,event_importance,search_observatory,area_raw,concept,solar_object_locator,hgc_boundcc,hrc_a,gs_galleryid,hgs_coord,ar_zurichcls,bound_ccnsteps,intenskurt,event_clippedtemporal,hee_x,hee_y,hee_z,cme_radiallinvelmax,cme_radiallinveluncert,rasterscantype,search_channelid,hgs_bbox,cme_radiallinvelmin,cme_radiallinvel,obs_lastprocessingdate,refs,event_coord
,,"arcsec,arcsec",,"deg,deg",,"arcsec,arcsec",,,,,"solRad,deg,solRad",,,,,cm,,"deg,deg",,,,,,,,,"deg,deg",,,,,,,,"arcsec,arcsec",,,,,,,,,,,deg,,deg,,"solRad,deg,solRad",,,,,,,,,"solRad,deg,solRad",,,,,deg,,,,,,,,,"deg,deg",,,,km / s,,,deg,,,deg,,,,,deg,,,,,,,,deg,,,,,,,,,"deg,deg",,,"deg,deg",,,,,,,,km / s,km / s,,,"deg,deg",km / s,km / s,,,
str1,str1,SkyCoord,str5,SkyCoord,float64,SkyCoord,float64,str1,int64,str28,SkyCoord,Time,str2,str2,float64,float64,str21,SkyCoord,str5,str4,float64,float64,float64,float64,str2,str1,SkyCoord,float64,str2,str33,str2,str2,object,str1,SkyCoord,str2,str14,str2,float64,float64,str36,float64,str6,str5,str2,float64,str2,float64,str35,SkyCoord,str1,str2,str2,float64,str1,str19,float64,str2,SkyCoord,float64,str5,float64,str19,float64,float64,float64,str5,int64,str19,str13,float64,str1,SkyCoord,float64,str2,str2,float64,str87,str6,float64,int64,float64,float64,str1,str2,float64,str5,float64,float64,int64,str2,str5,str2,str1,str5,float64,str2,str18,Time,float64,str5,float64,str3,str1,SkyCoord,int64,str1,SkyCoord,str2,int64,float64,str2,object,object,str1,float64,float64,str2,str5,SkyCoord,float64,float64,str2,object[4],object
,0,"-1814.4585430000443,-624.768179 .. -1814.4585430000443,-624.768179",false,"———,———",--,"-1709.8494320553727,-871.2118004358282",———,,--,thresh=0.30 factor=6 width=5,"2.0,207.0,1.0",2022-04-02 02:48:07.000,,CE,———,6e-05,http://sidc.be/cactus,"———,———",false,true,———,———,2.5,———,,,"———,———",———,,"C2 orange filter, C3 clear filter",,,None,3,"———,———",,cactus@sidc.be,,———,———,CACTus (Computer Aided CME Tracking),———,CACTus,LASCO,,30.0,,2.0,Royal Observatory of Belgium - SIDC,"2.0,199.0,1.0 .. 2.0,199.0,1.0",,,,———,,0.09225456339417859,———,,"———,———,1.0",--,false,———,2022-04-05T01:20:58,16.0,-871.2118004358282,-1709.8494320554128,LASCO,--,2022-04-05T01:21:17,robbrecht_eva,———,0,"———,———",———,,,28.0,ivo://helio-informatics.org/CE_CACTus(ComputerAidedCMETracking)_20220405_012058_4228.41,CACTus,125.0,--,———,109.0,,,———,false,0.32264599204063416,———,2,,false,,,c2 c3,2.0,1,1919.00888,2022-04-02 02:48:07.000,--,LASCO,———,CME,,"———,———",117,,"———,———",,--,———,,None,None,,238.0,18.700000762939453,,"C2,C3","———,———",180.0,187.0,,"{'ref_name': 'FRM_URL', 'ref_type': 'unknown', 'ref_url': 'http://sidc.be/cactus'} ..

In [53]:
hek_search[0][0]["hpc_bbox"]

<SkyCoord (Helioprojective: obstime=['2022-04-02T02:48:07.000' '2022-04-02T02:48:07.000'
 '2022-04-02T02:48:07.000' '2022-04-02T02:48:07.000'
 '2022-04-02T02:48:07.000'], rsun=695700.0 km, observer=<HeliographicStonyhurst Coordinate (obstime=['2022-04-02T02:48:07.000' '2022-04-02T02:48:07.000'
 '2022-04-02T02:48:07.000' '2022-04-02T02:48:07.000'
 '2022-04-02T02:48:07.000'], rsun=695700.0 km): (lon, lat, radius) in (deg, deg, AU)
    [(0., -6.49773403, 0.99942745), (0., -6.49773403, 0.99942745),
     (0., -6.49773403, 0.99942745), (0., -6.49773403, 0.99942745),
     (0., -6.49773403, 0.99942745)]>): (Tx, Ty) in arcsec
    [( -1814.458543,   -624.768179), ( -1571.960047,  -1100.698275),
     (-23579.400706, -16510.474121), (-27216.878142,  -9371.522692),
     ( -1814.458543,   -624.768179)]>